# Lab 1 — QDMI: from your laptop (classical node) to IQM Resonance (quantum node)

**Workshop block 3 · 30 min** · IQM HPC + Quantum workshop

HPC centres integrate accelerators through *stable interfaces*: your MPI code doesn't care which
interconnect vendor is underneath. The **Quantum Device Management Interface (QDMI)** brings the same idea to
quantum computers: one vendor-neutral C API with three function groups:

| Group | What it does | Typical calls |
|---|---|---|
| **Session** | authenticated connection to one device | `session_alloc`, `set_parameter(BASEURL, TOKEN, …)`, `session_init` |
| **Query** | device, site (qubit) and operation properties — topology, T1/T2, fidelities | `query_device_property`, `query_site_property`, `query_operation_property` |
| **Job** | submit a program, track it, fetch results | `create_device_job`, `job_submit`, `job_check/wait`, `job_get_results` |

**QDMI-on-IQM** (`github.com/iqm-finland/QDMI-on-IQM`, `pip install iqm-qdmi`) is IQM's production implementation
of that interface. It is the layer under IQM's HPC Integration Service, which runs IQM Radiance systems as Slurm nodes.

In this lab you will

1. install QDMI on your own machine and list the devices it can see,
2. run a program on the **local classical node** (MQT DDSIM simulator, behind the *same* QDMI interface),
3. point the *same code* at the **quantum node**: IQM Resonance in the cloud,
4. measure the **round-trip latency** of a classical↔quantum loop, the number that decides which hybrid algorithms are practical,
5. *(optional)* run it all again as **Slurm jobs** on a mini HPC cluster in Docker.

```
 Qiskit / your code ─► MQT Core QDMIBackend ─► QDMI driver ─┬─► DDSIM device library   (your laptop)
                                                            └─► IQM QDMI device library ─► IQM Server API ─► Resonance / Radiance
```

## 0 · Setup (do this before the workshop)

```bash
./setup_local.sh            # creates .venv with mqt-core[qiskit] + iqm-qdmi[qiskit] + jupyterlab
source .venv/bin/activate
cp env.example .env         # paste your Resonance token
```

Get a token at **https://resonance.meetiqm.com** → side panel → **Create token**. The free *Starter* tier
includes monthly credits; the `:mock` devices cost nothing and are perfect for this lab.

Uncomment the next cell if you are running in a fresh kernel without the setup script.

In [ ]:
# %pip install -q "mqt-core[qiskit]" "iqm-qdmi[qiskit]" matplotlib

In [ ]:
import json, os, time
from pathlib import Path

import mqt.core, qiskit
print("mqt-core", mqt.core.__version__, "| qiskit", qiskit.__version__)
try:
    from importlib.metadata import version
    print("iqm-qdmi", version("iqm-qdmi"))
except Exception as exc:
    print("iqm-qdmi not installed:", exc)

## 1 · Discover devices — no network, no libraries loaded yet

The builtin MQT Core QDMI driver reads *device manifests*: small JSON files that map a **stable device ID** to a
device library and its session defaults. Installing `iqm-qdmi` drops IQM manifests next to the ones MQT Core
ships (e.g. `mqt.ddsim.default`). Listing them only parses metadata.

In [ ]:
from mqt.core.qdmi import builtin_driver, ProgramFormat

device_ids = builtin_driver.registered_device_ids()
for d in device_ids:
    print(f"{d:<40} {'IQM (remote)' if d.lower().startswith('iqm') else 'local'}")

IQM_IDS = [d for d in device_ids if d.lower().startswith("iqm")]
LOCAL_ID = "mqt.ddsim.default"

> **HPC note.** Operators can publish a site-wide catalogue in `/etc/mqt-core/qdmi.json` (endpoints, presets) and
> users can override in `~/.config/mqt-core/qdmi.json` or a project-level `qdmi.json`. Credentials never go in the
> catalogue; they come from the environment, a tokens file, or a Slurm SPANK plugin.

## 2 · The classical node: run through QDMI on your laptop

We open **one** device by ID, query it, and submit an OpenQASM 2 program. These Python calls map 1:1 onto the C
calls in QDMI (`QDMI_device_session_init`, `QDMI_device_job_submit`, `QDMI_device_job_wait`, `QDMI_device_job_get_results`).

In [ ]:
def prop(obj, name):
    """QDMI devices may legally not support a property, so query defensively."""
    f = getattr(obj, name, None)
    try:
        return f() if callable(f) else f
    except Exception as exc:
        return f"<{exc.__class__.__name__}>"

local = builtin_driver.open_device(LOCAL_ID)
for p in ["name", "status", "qubits_num", "supported_program_formats"]:
    print(f"{p:<26} {prop(local, p)}")

In [ ]:
BELL_QASM2 = """OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];
h q[0];
cx q[0], q[1];
measure q -> c;
"""
t0 = time.perf_counter()
job = local.submit_job(BELL_QASM2, ProgramFormat.QASM2, 1000)
assert job.wait(60)
print(job.get_counts(), f"  ({time.perf_counter() - t0:.3f}s)")

## 3 · The quantum node: IQM Resonance through QDMI-on-IQM

Load your token into the environment. QDMI-on-IQM reads credentials from session parameters **or** environment
variables, which is why the same script later runs unchanged inside `sbatch`.

| QDMI session parameter | IQM meaning | Here |
|---|---|---|
| `BASEURL` | IQM Server endpoint | `IQM_BASE_URL` |
| `TOKEN` / `AUTHFILE` | API token or tokens file | `RESONANCE_API_KEY` / `IQM_TOKENS_FILE` |
| `CUSTOM1` / `CUSTOM2` | quantum computer ID / alias selector | `IQM_QC_ALIAS` → `custom2` |

In [ ]:
from getpass import getpass

env_file = Path(".env")
if env_file.exists():
    for line in env_file.read_text().splitlines():
        if line.strip() and not line.lstrip().startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.split("#")[0].strip())
if not (os.environ.get("RESONANCE_API_KEY") or os.environ.get("IQM_TOKENS_FILE")):
    os.environ["RESONANCE_API_KEY"] = getpass("IQM Resonance API token: ")

def iqm_session_kwargs():
    kw = {}
    if os.environ.get("IQM_BASE_URL"):
        kw["base_url"] = os.environ["IQM_BASE_URL"]
    if os.environ.get("RESONANCE_API_KEY"):
        kw["token"] = os.environ["RESONANCE_API_KEY"]
    elif os.environ.get("IQM_TOKENS_FILE"):
        kw["auth_file"] = os.environ["IQM_TOKENS_FILE"]
    if os.environ.get("IQM_QC_ALIAS"):
        kw["custom2"] = os.environ["IQM_QC_ALIAS"]
    return kw

print("IQM device IDs:", IQM_IDS)
# Pick a *mock* device first: same API, same queue semantics, zero credits.
IQM_ID = os.environ.get("IQM_DEVICE_ID") or next((d for d in IQM_IDS if "mock" in d), IQM_IDS[0] if IQM_IDS else None)
print("using:", IQM_ID)

In [ ]:
t0 = time.perf_counter()
iqm = builtin_driver.open_device(IQM_ID, **iqm_session_kwargs())
print(f"session init {time.perf_counter() - t0:.2f}s  (authentication + backend discovery + calibration fetch)")
for p in ["name", "status", "qubits_num", "needs_calibration", "supported_program_formats"]:
    print(f"{p:<26} {prop(iqm, p)}")

### 3a · Query the hardware: this is what a compiler or scheduler sees

Static data (topology, site IDs) and dynamic calibration data (T1/T2, fidelities) are cached at session
initialisation, so queries are local and cheap. A scheduler can use them to route around a degraded qubit.

In [ ]:
sites = prop(iqm, "sites") or []
cmap = prop(iqm, "coupling_map") or []
print(f"{len(sites)} sites, {len(cmap)} couplers")
for s in sites[:6]:
    print(f"  site {prop(s, 'index'):>3}  name={prop(s, 'name')}  T1={prop(s, 't1')}  T2={prop(s, 't2')}")
for op in (prop(iqm, "operations") or [])[:6]:
    print(f"  op {str(prop(op, 'name')):<8} qubits={prop(op, 'qubits_num')}  fidelity={prop(op, 'fidelity')}")

In [ ]:
import matplotlib.pyplot as plt

try:  # draw the coupling graph from QDMI data alone
    import networkx as nx
    G = nx.Graph()
    G.add_edges_from((prop(a, "index"), prop(b, "index")) for a, b in cmap)
    plt.figure(figsize=(5, 5)); nx.draw_kamada_kawai(G, with_labels=True, node_color="#21b5a8", font_color="white")
    plt.title(f"{IQM_ID}: coupling map from QDMI"); plt.show()
except ImportError:
    print("pip install networkx to draw the coupling graph")

### 3b · Submit a native IQM program (raw QDMI job)

IQM devices accept **IQM JSON** (and QIR). Native gates are `prx(θ, φ)` with angles in *turns* and `cz`.
A Bell pair is `RY(π/2)` on both qubits → `CZ` → `RY(−π/2)` on the target. We take the qubit pair straight from the
coupling map we just queried.

In [ ]:
a, b = cmap[0] if cmap else (None, None)
q0 = prop(a, "name") if a is not None and isinstance(prop(a, "name"), str) else "QB1"
q1 = prop(b, "name") if b is not None and isinstance(prop(b, "name"), str) else "QB2"
bell_iqm = json.dumps({"name": "bell", "instructions": [
    {"name": "prx", "locus": [q0], "args": {"angle_t": 0.25, "phase_t": 0.25}},
    {"name": "prx", "locus": [q1], "args": {"angle_t": 0.25, "phase_t": 0.25}},
    {"name": "cz", "locus": [q0, q1], "args": {}},
    {"name": "prx", "locus": [q1], "args": {"angle_t": 0.25, "phase_t": 0.75}},
    {"name": "measure", "locus": [q0, q1], "args": {"key": "m"}},
]})
t0 = time.perf_counter()
job = iqm.submit_job(bell_iqm, ProgramFormat.IQM_JSON, 500)
print("job id:", prop(job, "id"), "| status:", prop(job, "check"))
assert job.wait(600), "job did not finish"
print(job.get_counts(), f"  ({time.perf_counter() - t0:.1f}s submit→result)")

### 3c · The same thing, but from Qiskit

Most users will never write IQM JSON. The MQT Core `QDMIBackend` is a Qiskit `BackendV2` built **entirely from
QDMI queries**: its `Target` (native gates, coupling map, fidelities) comes from the device. Transpile against it,
run, done. Switching hardware vendor means changing a device ID, not a provider package.

In [ ]:
from mqt.core.plugins.qiskit import QDMIBackend
from qiskit import QuantumCircuit, transpile

def bell():
    qc = QuantumCircuit(2, 2); qc.h(0); qc.cx(0, 1); qc.measure([0, 1], [0, 1]); return qc

for dev_id, kw in [(LOCAL_ID, {}), (IQM_ID, iqm_session_kwargs())]:
    backend = QDMIBackend.from_device_id(dev_id, **kw)
    tqc = transpile(bell(), backend=backend, optimization_level=1, seed_transpiler=1)
    t0 = time.perf_counter()
    counts = backend.run(tqc, shots=500).result().get_counts()
    print(f"{dev_id:<28} native ops={dict(tqc.count_ops())}  counts={counts}  {time.perf_counter() - t0:.1f}s")

## 4 · A classical↔quantum loop: measure your coupling latency

Variational algorithms (VQE, QAOA, the VQE inside QSCI) alternate: the classical node proposes parameters, the
quantum node returns expectation values. Here the "optimiser" is a simple sweep of θ in `RY(θ)` and we time every
round trip. Compare local vs. Resonance, and remember the number for the QSCI lab.

In [ ]:
from qiskit.circuit import Parameter
import numpy as np

def latency_sweep(dev_id, kw, steps=6, shots=300):
    backend = QDMIBackend.from_device_id(dev_id, **kw)
    th = Parameter("theta"); qc = QuantumCircuit(1, 1); qc.ry(th, 0); qc.measure(0, 0)
    tqc = transpile(qc, backend=backend, optimization_level=1)         # transpile ONCE, bind many times
    thetas, zs, rtts = np.linspace(0, np.pi, steps), [], []
    for t in thetas:
        t0 = time.perf_counter()
        c = backend.run(tqc.assign_parameters({th: t}), shots=shots).result().get_counts()
        rtts.append(time.perf_counter() - t0)
        zs.append((c.get("0", 0) - c.get("1", 0)) / shots)
    return thetas, np.array(zs), np.array(rtts)

results = {LOCAL_ID: latency_sweep(LOCAL_ID, {}), IQM_ID: latency_sweep(IQM_ID, iqm_session_kwargs())}

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for dev_id, (t, z, r) in results.items():
    ax[0].plot(t, z, "o-", label=dev_id); ax[1].bar(dev_id, r.mean(), yerr=r.std())
    print(f"{dev_id:<28} mean round trip {r.mean():.2f}s")
ax[0].plot(t, np.cos(t), "k--", label="cos θ (exact)"); ax[0].set_xlabel("θ"); ax[0].set_ylabel("⟨Z⟩"); ax[0].legend()
ax[1].set_ylabel("seconds per iteration"); ax[1].set_yscale("log"); plt.tight_layout(); plt.show()

**Discuss.** A 100-iteration VQE with ~5 measurement circuits per iteration means ~500 round trips.
At cloud latency that is minutes to hours of *waiting*; inside the HPC centre (on-prem Radiance as a Slurm node,
job submitted from the QAN) it shrinks; with mid-circuit feedback or QEC decoding it must reach **microseconds**.
That gradient is the story of the last two workshop blocks.

## 5 · (Optional) Same script, as Slurm jobs on a mini HPC centre

`slurm/start_playground.sh` builds the MQT Core Slurm playground in Docker: `login`, `controller`, two classical
`node`s (**compute** partition), two quantum access nodes `qan` (**quantum** partition) and SlurmDBD accounting.
QDMI devices are scheduled as **Slurm licences named after the device ID**.

```bash
# on your laptop
./slurm/start_playground.sh                        # first build takes a while
docker compose -f ~/qdmi-workshop/core/examples/slurm/compose.yml exec --user 10000:10000 login bash --login

# inside the login node
sinfo                                              # compute + quantum partitions
scontrol show licenses                             # mqt.ddsim.default, iqm.* ...
export RESONANCE_API_KEY=...                       # exported into jobs by default
cd /jobs
sbatch --wait --licenses=mqt.ddsim.default bell.sbatch && cat qdmi-bell-*.out

# chain: quantum stage on a QAN  ->  classical analysis on a compute node (holds no quantum licence)
QID=$(sbatch --parsable --licenses=iqm.garnet.mock bell.sbatch)
sbatch --dependency=afterok:$QID --export=ALL,BELL_JOB=$QID analyze.sbatch
squeue --me ; sacct -X --format=JobID,JobName,Partition,State,Elapsed,AllocTRES%60
```

Notice what did **not** change: `qdmi_demo.py` is the same file you ran on your laptop. Only the launch method
and the injected environment differ. That is the operational argument for a standard device interface.

### Three deployment scenarios, one code path

| Scenario | Coupling | Who injects endpoint + credentials | Good for |
|---|---|---|---|
| **Cloud QPU** (this lab) | loose | the user (`.env`, `sbatch --export`) | onboarding, development, testing pipelines |
| **On-prem QPU, working-node mode** | tight | user's job environment; QPU is a Slurm partition | co-scheduling, iterative hybrid jobs |
| **SPANK plugin** | tight + automated | the operator, in the Slurm prolog | production, credential rotation, site policy |

### Troubleshooting
* `registered_device_ids()` shows no `iqm.*`: `iqm-qdmi` is not in the active environment.
* Session init fails with an auth error: token not exported, expired, or missing `IQM_QC_ALIAS` for your account.
* `UnsupportedOperationError` from `QDMIBackend.run`: transpile against `backend` (target = native gates) first.
* Jobs pending with reason `Licenses` in Slurm: the device licence is exhausted or the availability monitor marked the device down.

**Next:** Lab 2 (Qrisp) writes algorithms at a higher level; Lab 3 runs QSCI as a two-stage Slurm pipeline through QDMI.